# Capstone – Gestión de Telecomunicaciones II  
## Proyecto 1: 5G FWA Rural (Escenario Base Uniforme)

Dataset y escenario numérico **uniforme** para todos los grupos.

**Archivos (carpeta `data/`):**
- `demand_points.csv` (puntos de demanda con hogares y suscriptores iniciales)
- `candidate_sites.csv` (sitios candidatos)
- `link_budget_matrices.npz` (dist_km, pathloss_db, rsrp_dbm)
- `scenario_params.json` (parámetros del escenario)

Incluye:
- Modelo simplificado de SINR (reuse-1) y capacidad
- Baseline de selección de sitios
- Plantilla de búsqueda local para Sprint 4


In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")
demand = pd.read_csv(DATA_DIR/"demand_points.csv")
sites = pd.read_csv(DATA_DIR/"candidate_sites.csv")
mat = np.load(DATA_DIR/"link_budget_matrices.npz")
dist_km = mat["dist_km"]
pathloss_db = mat["pathloss_db"]
rsrp_dbm = mat["rsrp_dbm"]
params = json.loads((DATA_DIR/"scenario_params.json").read_text())

print("Demand points:", len(demand))
print("Candidate sites:", len(sites))
print("RSRP matrix:", rsrp_dbm.shape)
demand.head()


In [ ]:
BW_HZ = params["radio"]["bandwidth_mhz"]*1e6
NF_DB = params["radio"]["noise_figure_db"]
SE_CAP = 6.0

def dbm_to_w(dbm):
    return 10**((dbm-30)/10)

def thermal_noise_dbm(bw_hz, nf_db):
    return -174 + 10*np.log10(bw_hz) + nf_db

NOISE_DBM = thermal_noise_dbm(BW_HZ, NF_DB)
noise_w = dbm_to_w(NOISE_DBM)
print("Noise (dBm):", NOISE_DBM)


In [ ]:
def evaluate_plan(selected_sites, reuse1=True):
    sel = np.array(selected_sites, dtype=int)
    if len(sel)==0:
        return None

    rsrp_sel = rsrp_dbm[:, sel]  # [DP,S]
    best_local = np.argmax(rsrp_sel, axis=1)
    Pr_serv_dbm = rsrp_sel[np.arange(rsrp_sel.shape[0]), best_local]
    Pr_serv_w = dbm_to_w(Pr_serv_dbm)

    if reuse1 and len(sel)>1:
        Pr_all_w = dbm_to_w(rsrp_sel)
        interf_w = Pr_all_w.sum(axis=1) - Pr_serv_w
    else:
        interf_w = np.zeros_like(Pr_serv_w)

    sinr = Pr_serv_w / (interf_w + noise_w)
    sinr_db = 10*np.log10(sinr + 1e-12)

    se = np.log2(1+sinr)
    se = np.minimum(se, SE_CAP)
    cap_bps = BW_HZ * se

    subs = demand["subscribers_initial"].to_numpy()
    conc = params["demand"]["busy_hour_concurrency"]
    target_mbps = params["demand"]["target_throughput_mbps_per_subscriber"]
    demand_bps = subs * conc * target_mbps * 1e6

    satisfied = cap_bps >= demand_bps
    hh = demand["households"].to_numpy()

    pop_cov = float((hh*(cap_bps>0)).sum()/hh.sum())
    sat_frac = float((hh*satisfied).sum()/hh.sum())

    metrics = {
        "sites_deployed": int(len(sel)),
        "pop_covered_frac": pop_cov,
        "capacity_satisfied_frac": sat_frac,
        "avg_sinr_db": float(np.mean(sinr_db)),
        "p05_sinr_db": float(np.quantile(sinr_db, 0.05)),
        "avg_se_bphz": float(np.mean(se)),
    }
    return metrics

# Example
evaluate_plan([0,1,2,3,4])


In [ ]:
def greedy_by_best_rsrp(S=8, thr_dbm=-95):
    score = (rsrp_dbm > thr_dbm).sum(axis=0)
    return list(np.argsort(-score)[:S])

def objective(m, w_cov=1.0, w_cap=1.0, w_sites=0.15):
    return w_cov*m["pop_covered_frac"] + w_cap*m["capacity_satisfied_frac"] - w_sites*(m["sites_deployed"]/params["planning"]["max_sites_to_deploy"])

def local_search(initial_sites, iters=400, S=8, seed=0):
    rng = np.random.default_rng(seed)
    all_sites = np.arange(len(sites))

    best = sorted(list(dict.fromkeys(initial_sites)))[:S]
    best_m = evaluate_plan(best)
    best_J = objective(best_m)

    for _ in range(iters):
        cand = best.copy()
        out = int(rng.integers(0, S))
        cand[out] = int(rng.choice(all_sites))
        cand = sorted(list(set(cand)))
        if len(cand)>S:
            cand = cand[:S]
        while len(cand)<S:
            cand.append(int(rng.choice(all_sites)))
            cand = sorted(list(set(cand)))
            if len(cand)>S:
                cand = cand[:S]

        m = evaluate_plan(cand)
        J = objective(m)
        if J > best_J:
            best, best_m, best_J = cand, m, J

    return best, best_m, best_J

init = greedy_by_best_rsrp(S=8)
best_sites, best_m, best_J = local_search(init, iters=600, S=8, seed=1)
print("Init:", init, evaluate_plan(init))
print("Best:", best_sites, best_m, best_J)
